# BÁO CÁO MILESTONE 3: REFERENCE MATCHING PIPELINE
## Phần phụ trách: Auto-Labeling & Feature Engineering
**Người thực hiện:** Hoàng (Nhóm 11 - NMKHDL)  
**Mục tiêu đồ án:** Ghép nối các mục trích dẫn BibTeX (Milestone 2) với siêu dữ liệu bài báo arXiv tham chiếu (Milestone 1 - Semantic Scholar API) thông qua pipeline Học Máy, đánh giá bằng chỉ số **MRR@5**.


## 1. Tổng quan & Khung bài toán (Problem Framing)

### 1.1. Không gian ứng viên (Candidate Space)
Trong mỗi ấn phẩm khoa học (publication) chứa $m$ mục BibTeX trích dẫn và $n$ bài báo tham chiếu đã cào có arXiv ID:
- Không gian ứng viên tạo thành ma trận nhị phân kích thước $m \times n$.
- Mỗi ô $(i, j)$ mang nhãn dương (1 - **yes**) nếu mục BibTeX $i$ ứng với tham chiếu $j$, ngược lại mang nhãn âm (0 - **no**).
- Với mỗi mục BibTeX, chỉ có tối đa một tham chiếu đúng (positive label). Nếu mục BibTeX không tương ứng với bài arXiv nào trong $n$ bài tham chiếu thì toàn bộ hàng mang giá trị 0 (all-negative row).

### 1.2. Định dạng bài toán Học Máy
Chúng tôi định hình bài toán dưới dạng **Binary Classification / Pointwise Ranking**:
- **Đầu vào:** Mỗi cặp ứng viên $(BibTeX_i, Reference_j)$ được biểu diễn bằng một vector đặc trưng đa chiều $x_{ij} \in \mathbb{R}^D$.
- **Mục tiêu:** Mô hình học máy (như Logistic Regression, LightGBM, Random Forest do Tân phụ trách) dự đoán xác suất $P(y_{ij} = 1 \mid x_{ij})$.
- **Dự đoán Top-5 & Đánh giá:** Với mỗi mục BibTeX $i$, sắp xếp các ứng viên $j$ theo xác suất giảm dần để lấy top 5 ứng viên ($cand_1 \dots cand_5$), tính chỉ số **MRR@5**:
$$\text{MRR@5} = \frac{1}{|Q|} \sum_{i=1}^{|Q|} \frac{1}{\text{rank}_i}$$


## 2. Phương pháp Gán nhãn Tự động (Auto-Labeling Pipeline)

Theo đặc tả của Milestone 3:
- Pool gán nhãn tự động chiếm ít nhất **10%** số publication ngoài tập thủ công (và tối thiểu $2k = 8$ publications với nhóm 4 người).
- Thuật toán gán nhãn tự động phải gán nhãn cho **toàn bộ** các mục BibTeX trong publication được chọn.

### 2.1. Các quy tắc Heuristics có độ chuẩn xác cao (High-Precision Rules)
1. **Rule 1 - Direct arXiv ID Match (Độ tin cậy 1.0):** Trích xuất định danh arXiv qua regex từ các trường `eprint`, `url`, `note` của BibTeX và so sánh trực tiếp với `ref_arxiv_id`.
2. **Rule 2 - Exact Normalized Title + Metadata Verification (Độ tin cậy 0.98):** Chuẩn hóa tiêu đề (loại bỏ ký tự LaTeX, dấu câu, đưa về chữ thường). Nếu tiêu đề trùng khớp hoàn toàn và có ít nhất 1 tác giả trùng hoặc năm xuất bản lệch không quá 1 năm.
3. **Rule 3 - Fuzzy Title Similarity & Margin Verification (Độ tin cậy 0.90 - 0.95):** Dùng `Token Sort Ratio` / `Levenshtein Ratio` $\ge 90\%$. Đồng thời ứng viên đứng đầu phải vượt trội ứng viên thứ hai một khoảng cách (margin) an toàn để tránh nhập nhằng.
4. **Trường hợp âm tính (All-negative row):** Nếu không có ứng viên nào vượt qua các ngưỡng trên, mục BibTeX được gán nhãn `None`.


In [ ]:
# Import các module đã xây dựng trong src/model
import sys
import pandas as pd
import numpy as np

from src.model.data_cleaning import clean_bibtex_table, clean_references_table
from src.model.auto_labeler import AutoReferenceMatcher, select_auto_label_publications
from src.model.feature_engineering import FeatureExtractor

print("Modules imported successfully!")


In [ ]:
# Demo quy trình Auto-Labeling trên một publication mẫu
matcher = AutoReferenceMatcher()

# Dữ liệu BibTeX mẫu
sample_bib = pd.DataFrame([
    {
        "arxiv_id": "2411.13807",
        "citation_key": "vaswani2017attention",
        "title": "Attention Is All You Need",
        "authors": "A. Vaswani, N. Shazeer, N. Parmar",
        "year": 2017,
        "eprint": "1706.03762"
    },
    {
        "arxiv_id": "2411.13807",
        "citation_key": "devlin2018bert",
        "title": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "authors": "J. Devlin, M. Chang, K. Lee, K. Toutanova",
        "year": 2018,
        "url": ""
    },
    {
        "arxiv_id": "2411.13807",
        "citation_key": "external_math_book",
        "title": "Introduction to Topology and Modern Analysis",
        "authors": "G. F. Simmons",
        "year": 1963
    }
])

# Dữ liệu References cào được từ Semantic Scholar
sample_ref = pd.DataFrame([
    {
        "citing_arxiv_id": "2411.13807",
        "ref_arxiv_id": "1706.03762",
        "title": "Attention is all you need",
        "authors": ["Ashish Vaswani", "Noam Shazeer"],
        "year": 2017
    },
    {
        "citing_arxiv_id": "2411.13807",
        "ref_arxiv_id": "1810.04805",
        "title": "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "authors": ["Jacob Devlin", "Ming-Wei Chang"],
        "year": 2018
    },
    {
        "citing_arxiv_id": "2411.13807",
        "ref_arxiv_id": "2005.14165",
        "title": "Language Models are Few-Shot Learners",
        "authors": ["Tom B. Brown"],
        "year": 2020
    }
])

clean_bib = clean_bibtex_table(sample_bib)
clean_ref = clean_references_table(sample_ref)

labels = matcher.label_publication(clean_bib, clean_ref)
for r in labels:
    print(f"Key: {r.citation_key:<22} -> Match: {str(r.groundtruth_arxiv_id):<12} | Method: {r.method:<26} | Conf: {r.confidence}")


## 3. Kỹ nghệ đặc trưng (Feature Engineering)

Để mô hình học máy phân biệt chính xác giữa cặp đúng và các cặp nhiễu, chúng tôi thiết kế **5 nhóm đặc trưng** chính với tổng cộng 23 chiều:

### 3.1. Nhóm đặc trưng tương đồng Tiêu đề (Title Features)
- `sim_title_exact`: Cờ kiểm tra tiêu đề trùng khớp tuyệt đối (1/0).
- `sim_title_ratio`: Độ tương đồng chuỗi Levenshtein ratio.
- `sim_title_token_sort`: Độ tương đồng sau khi sắp xếp từ khóa (bỏ qua khác biệt thứ tự từ hoặc dấu câu).
- `sim_title_token_set`: Xử lý trường hợp một bên chứa phụ đề (subtitle) hoặc viết hoa viết tắt.
- `sim_title_jaccard_words`: Tỉ lệ Jaccard trên tập các từ vựng tiêu đề.
- `sim_title_char_3gram`: Tỉ lệ Jaccard trên 3-gram ký tự (rất hiệu quả với lỗi chính tả hoặc phiên âm).
- `feat_title_length_diff`: Chênh lệch độ dài ký tự giữa 2 tiêu đề.

### 3.2. Nhóm đặc trưng Tác giả (Author Features)
- `sim_author_exact`: Cờ trùng khớp danh sách tác giả chuẩn hóa.
- `sim_author_jaccard`: Tỉ lệ giao/hợp giữa danh sách họ tên tác giả.
- `feat_author_first_match`: Cờ trùng họ của tác giả đầu tiên (First author surname match - tín hiệu rất mạnh trong văn hiến khoa học).
- `feat_author_count_diff`: Chênh lệch số lượng tác giả được liệt kê.

### 3.3. Nhóm đặc trưng Năm xuất bản (Year Signals)
- `feat_year_exact`: Cờ trùng năm xuất bản.
- `feat_year_diff`: Độ lệch số năm $|Year_{bib} - Year_{ref}|$.
- `feat_year_missing`: Cờ báo thiếu năm ở một trong hai phía.

### 3.4. Nhóm đặc trưng Định danh (Identifier Signals)
- `feat_id_exact_match`: Trùng khớp tuyệt đối mã định danh arXiv trích xuất được.
- `feat_has_bib_arxiv_id`: Cờ nhận biết mục BibTeX có thông tin arXiv hay không.

### 3.5. Nhóm đặc trưng Ngữ cảnh & Cây phân cấp (Hierarchy & Group-Relative)
- Tận dụng cây phân cấp Milestone 2: `feat_citation_depth` (độ sâu section chứa trích dẫn), `feat_citation_frequency` (tần suất trích dẫn trong bài).
- Đặc trưng thứ hạng ứng viên: `feat_title_sim_rank` (thứ hạng tương đồng của ứng viên này so với các ứng viên khác trong cùng mục BibTeX), `feat_title_sim_diff_to_max` (khoảng cách điểm số tới ứng viên cao nhất).


In [ ]:
# Trích xuất ma trận đặc trưng Candidate Matrix
extractor = FeatureExtractor()
labels_df = pd.DataFrame([
    {"arxiv_id": "2411.13807", "citation_key": "vaswani2017attention", "groundtruth_arxiv_id": "1706.03762"},
    {"arxiv_id": "2411.13807", "citation_key": "devlin2018bert", "groundtruth_arxiv_id": "1810.04805"},
    {"arxiv_id": "2411.13807", "citation_key": "external_math_book", "groundtruth_arxiv_id": None}
])

matrix = extractor.create_candidate_matrix(sample_bib, sample_ref, labels_df)
print(f"Candidate matrix shape: {matrix.shape}")
print(matrix[['citation_key', 'candidate_ref_id', 'sim_title_ratio', 'sim_title_token_sort', 'feat_author_first_match', 'feat_year_diff', 'label']].head(6))


## 4. Phân tích Tương quan & Đóng góp của Đặc trưng (Feature Analysis)

Dưới đây là phân tích mức độ phân tách của các nhóm đặc trưng chính giữa các cặp Positive (label=1) và Negative (label=0):


In [ ]:
# Thống kê giá trị trung bình của các features theo nhãn
numeric_cols = [c for c in matrix.columns if c.startswith(('sim_', 'feat_')) and c in matrix]
mean_by_label = matrix.groupby('label')[numeric_cols].mean().T
mean_by_label.columns = ['Negative Pairs (0)', 'Positive Pairs (1)']
mean_by_label['Separation (Diff)'] = mean_by_label['Positive Pairs (1)'] - mean_by_label['Negative Pairs (0)']
print(mean_by_label.sort_values(by='Separation (Diff)', ascending=False).head(10))


## 5. Kết luận & Bàn giao dữ liệu cho thành viên trong nhóm

1. **Phần Auto-Labeling:**
   - Đã đóng gói thành module độc lập `src.model.auto_labeler`, sẵn sàng nhận dữ liệu cào hoàn chỉnh để tạo ground truth cho tập Train/Val/Test theo đúng tỷ lệ đồ án.
2. **Phần Feature Engineering:**
   - Đã xây dựng hoàn thiện `FeatureExtractor` trong `src.model.feature_engineering` với 23 features đa dạng.
   - Hỗ trợ xuất trực tiếp định dạng Parquet / CSV để Tân nạp vào mô hình LightGBM / Ranking.
3. **Phần Báo cáo & Poster:**
   - Toàn bộ cơ sở lý thuyết, công thức tính toán và phân tích tương quan đã được lưu trữ sẵn sàng để trình bày trên Poster A2.
